# Lab 04 Challenge: worked solution

The challenge: stop an agent on a cost budget as well as a turn cap. This notebook runs on its own. It repeats the
config cell, the three lookups and their schemas, Part E's `run_tool` and Part B's `print_turns` from the lab, then
writes `run_agent_on_budget` and runs it on Puja Srivastava's question with two budgets.

## Setup: the pieces from the lab

In [1]:
from pathlib import Path
import os
from dotenv import dotenv_values

# The repository folder is the first folder, from here upwards, that holds models.env
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())

# Models and prices come from models.env; a non-empty value in .env replaces one of them
settings = dotenv_values(ROOT / "models.env")
local = dotenv_values(ROOT / ".env")
settings.update({name: value for name, value in local.items() if name in settings and value})

# The key comes from the environment or .env, and is never printed
os.environ.setdefault("OPENAI_API_KEY", local.get("OPENAI_API_KEY") or "")
assert os.environ["OPENAI_API_KEY"], "Put OPENAI_API_KEY in .env at the repository folder (copy .env.example)."

MODEL = settings["MODEL_DEFAULT"]
EFFORT = settings["REASONING_EFFORT"] or None
print("model:", MODEL, "| reasoning effort:", EFFORT, "| key found: yes")

model: gpt-6-luna | reasoning effort: none | key found: yes


In [2]:
from openai import OpenAI, omit

client = OpenAI()
MAX_OUTPUT = 300
PRICE_IN = float(settings["PRICE_INPUT_PER_MILLION"])
PRICE_OUT = float(settings["PRICE_OUTPUT_PER_MILLION"])

The three lookups on the shop's database, opened read-only, unchanged from Part A.

In [3]:
import json
import sqlite3

DB_PATH = ROOT / "data" / "chinook.db"
conn = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True, check_same_thread=False)


def find_customer(name):
    rows = conn.execute(
        "SELECT CustomerId, FirstName || ' ' || LastName, Country FROM customers "
        "WHERE FirstName || ' ' || LastName = ?", (name,)).fetchall()
    return [{"customer_id": r[0], "name": r[1], "country": r[2]} for r in rows]


def get_invoices(customer_id):
    count, total = conn.execute(
        "SELECT COUNT(*), ROUND(SUM(Total), 2) FROM invoices WHERE CustomerId = ?", (customer_id,)).fetchone()
    return {"customer_id": customer_id, "invoice_count": count, "total": total}


def get_support_rep(customer_id):
    row = conn.execute(
        "SELECT e.FirstName || ' ' || e.LastName, e.Email FROM customers c "
        "JOIN employees e ON e.EmployeeId = c.SupportRepId WHERE c.CustomerId = ?", (customer_id,)).fetchone()
    if row is None:
        raise ValueError(f"no customer with id {customer_id}")
    return {"customer_id": customer_id, "support_rep": row[0], "email": row[1]}

Their schemas, the dictionary of functions our code allows, and the system message, unchanged from Part A.

In [4]:
def tool_schema(name, description, properties):
    return {
        "type": "function",
        "function": {
            "name": name,
            "description": description,
            "strict": True,
            "parameters": {
                "type": "object",
                "properties": properties,
                "required": list(properties),
                "additionalProperties": False,
            },
        },
    }


TOOLS = [
    tool_schema("find_customer", "Find a shop customer by full name. Returns a list of matches with their customer_id.",
                {"name": {"type": "string", "description": "First and last name, for example 'Ana Silva'"}}),
    tool_schema("get_invoices", "Count a customer's invoices and add up their total in US dollars.",
                {"customer_id": {"type": "integer"}}),
    tool_schema("get_support_rep", "Name the support rep who looks after a customer.",
                {"customer_id": {"type": "integer"}}),
]

print([t['function']['name'] for t in TOOLS])

['find_customer', 'get_invoices', 'get_support_rep']


In [5]:
FUNCTIONS = {
    "find_customer": find_customer,
    "get_invoices": get_invoices,
    "get_support_rep": get_support_rep,
}

SYSTEM = ("You help the support reps of a music shop. Look up every fact with the tools; never answer from memory. "
          "Answer in one or two sentences.")
QUESTION = "How many invoices does Helena Holý have, what do they add up to, and who is her support rep?"
print(list(FUNCTIONS))

['find_customer', 'get_invoices', 'get_support_rep']


Part E's `run_tool`, which returns any error as text, and Part B's `print_turns`.

In [6]:
def run_tool(call):
    if call.function.name not in FUNCTIONS:
        return f"Error: there is no tool named {call.function.name}"
    try:
        arguments = json.loads(call.function.arguments)
        result = FUNCTIONS[call.function.name](**arguments)
    except Exception as error:
        return f"Error: {type(error).__name__}: {error}"
    return json.dumps(result, ensure_ascii=False)


def print_turns(turns):
    print(f"{'turn':>4}  {'finish_reason':13}  {'sent':>4}  {'prompt':>6}  {'output':>6}  tool calls")
    for t in turns:
        calls = ", ".join(t["tool_calls"]) or "-"
        print(f"{t['turn']:>4}  {t['finish_reason']:13}  {t['messages_sent']:>4}  "
              f"{t['prompt_tokens']:>6}  {t['completion_tokens']:>6}  {calls}")

## The budget loop

`run_agent_on_budget` is `run_agent` with three additions:

1. `spent` adds up the cost of every call so far, input tokens at the input price and output tokens at the output
   price.
2. `last_cost` is the cost of the call just made. It is our estimate of the next call, because the next prompt is the
   last prompt plus what was appended, so it is at least as long.
3. Before every call except the first (`if turns and ...`: an empty list is false), the loop checks whether
   `spent + last_cost` would pass the budget, and if so stops with `"budget"` and no answer. The first call is never
   refused: there is no last call to estimate from.

Each turn record also keeps its own cost and the running total, so the table shows where the money went.

In [7]:
def run_agent_on_budget(question, max_turns, budget):
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]
    turns, spent, last_cost = [], 0.0, 0.0
    for turn in range(1, max_turns + 1):
        if turns and spent + last_cost > budget:
            return None, "budget", turns, spent
        response = client.chat.completions.create(
            model=MODEL, messages=messages, tools=TOOLS,
            max_completion_tokens=MAX_OUTPUT, reasoning_effort=EFFORT or omit,
        )
        choice, usage = response.choices[0], response.usage
        last_cost = (usage.prompt_tokens * PRICE_IN + usage.completion_tokens * PRICE_OUT) / 1_000_000
        spent += last_cost
        turns.append({"turn": turn, "finish_reason": choice.finish_reason, "messages_sent": len(messages),
                      "prompt_tokens": usage.prompt_tokens, "completion_tokens": usage.completion_tokens,
                      "tool_calls": [f"{c.function.name}({c.function.arguments})" for c in choice.message.tool_calls or []],
                      "cost": last_cost, "spent": spent})
        messages.append(choice.message)
        if choice.finish_reason != "tool_calls":
            return choice.message.content, choice.finish_reason, turns, spent
        for call in choice.message.tool_calls:
            messages.append({"role": "tool", "tool_call_id": call.id, "content": run_tool(call)})
    return None, "turn cap", turns, spent

## Puja's question with a budget of 0.0002 dollars

First the generous budget. Puja Srivastava is customer 59: 6 invoices, 36.64 US dollars, rep Jane Peacock.

In [8]:
PUJA = "How many invoices does Puja Srivastava have, what do they add up to, and who is her support rep?"

answer, stop_reason, turns, spent = run_agent_on_budget(PUJA, max_turns=6, budget=0.0002)
print_turns(turns)
for t in turns:
    print(f"turn {t['turn']}: cost ${t['cost']:.7f}, spent so far ${t['spent']:.7f}")
print("stopped by:", stop_reason, "|", answer)

turn  finish_reason  sent  prompt  output  tool calls
   1  tool_calls        2     262      22  find_customer({"name":"Puja Srivastava"})
   2  tool_calls        4     321      51  get_invoices({"customer_id": 59}), get_support_rep({"customer_id": 59})
   3  stop              7     436      26  -
turn 1: cost $0.0000372, spent so far $0.0000372
turn 2: cost $0.0000576, spent so far $0.0000948
turn 3: cost $0.0000566, spent so far $0.0001514
stopped by: stop | Puja Srivastava has 6 invoices totaling $36.64. Her support rep is Jane Peacock.


Three turns, the same shape as Helena's run in the lab: `find_customer`, then the invoices and the rep together, then the answer, with the facts right (6 invoices, 36.64 US dollars, Jane Peacock). Both checks passed: before turn 2 the loop compared 0.0000372 + 0.0000372 = 0.0000744, and before turn 3 it compared 0.0000948 + 0.0000576 = 0.0001524, both under 0.0002. The run spent $0.0001514 in all.

## The same question with a budget of 0.0001 dollars

In [9]:
answer, stop_reason, turns, spent = run_agent_on_budget(PUJA, max_turns=6, budget=0.0001)
print_turns(turns)
for t in turns:
    print(f"turn {t['turn']}: cost ${t['cost']:.7f}, spent so far ${t['spent']:.7f}")
print("stopped by:", stop_reason, "| answer:", answer, f"| spent ${spent:.7f}")

turn  finish_reason  sent  prompt  output  tool calls
   1  tool_calls        2     262      22  find_customer({"name":"Puja Srivastava"})
   2  tool_calls        4     321      51  get_invoices({"customer_id": 59}), get_support_rep({"customer_id": 59})
turn 1: cost $0.0000372, spent so far $0.0000372
turn 2: cost $0.0000576, spent so far $0.0000948
stopped by: budget | answer: None | spent $0.0000948


The first two turns are the same as in the first run. Before turn 3 the check is 0.0000948 + 0.0000576 = 0.0001524, more than 0.0001, so the loop stops with `"budget"` and no answer, having spent $0.0000948. The call the budget refused is the one that would have written the answer: both tools had already run, and their results sat in the list unread. Like the turn cap, a budget is an exit the caller has to handle, not an answer.

## When the estimate is too low

The estimate is the last call's cost. The next call's **input** is never smaller, since its prompt is the last prompt
plus what was appended, but the estimate leaves out what was appended (here the tool results, a few dozen tokens),
and the next call's **output** can be longer than the last one's: a two-sentence answer after a short tool call. So
the estimate can fall short in two cases:

- a tool returns a long result (a page of invoices, a document), which the next call sends as input;
- the next reply is longer than the last, for example the final answer after a short tool call.

Both happened in these runs. Turn 2's estimate was turn 1's cost, $0.0000372, but turn 2 cost $0.0000576: its reply held two tool calls instead of one. Turn 3's estimate, $0.0000576, was close to its real cost in the first run, $0.0000566, because the answer was short. A stricter guard prices the next call at its worst case: the last prompt plus the appended
messages at the input price, and the full `MAX_OUTPUT` of 300 tokens at the output price
($300 / 10^6 \times 0.50 = 0.00015$ dollars for the output alone). That never lets a call pass the budget, at the
price of refusing calls that would have fitted. Which one to use is a choice: a hard limit, or a limit that is
usually right and cheaper to keep.